# Charge Category Classification

Goal: take the raw `Charge Type` / `Charge Description` labels from `trackingdetail Table.csv` (1,890 / 2,968 distinct raw values in the sample data) and roll them up into a small taxonomy: **5-7 major categories, each with up to 10 subcategories**.

**Approach**: rule-based keyword/regex classification, with a lightweight TF-IDF fuzzy-match fallback for anything the rules miss. No transformer/embedding model is used, on purpose:

- The vocabulary here is a *finite domain vocabulary* (carrier billing terms across a handful of languages), not open-ended natural language — keyword rules cover the large majority of it directly.
- Rule-based mapping is fully auditable: for any classified row you can point to the exact pattern that matched it. That matters for billing/finance data.
- Classification runs on the **unique (Charge Type, Charge Description) combinations**, not per row — there are only a few thousand of those even in the sample. This is what makes the approach reusable: production data with millions of rows still only has a few thousand distinct label combinations, so the same mapping table (or the same classifier re-run on new unique labels) scales without change.
- No heavy dependencies (torch / sentence-transformers) — just `scikit-learn`, which is already installed.

If, once applied to real company data, coverage from rules + TF-IDF fallback isn't high enough, an embedding-based clustering pass is a reasonable upgrade path — but it's not needed to get started.

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
ENC = "cp1252"

## Load charge-level data

In [ ]:
detail = pd.read_csv(
    "Database report1788285968903_trackingdetail Table.csv",
    encoding=ENC,
    usecols=["Charge Type", "Charge Description", "Charge Value"],
    low_memory=False,
)
print(detail.shape)
detail.head()

## Taxonomy: 6 substantive major categories (≤10 subcategories each) + 1 catch-all

Patterns are lowercase regex fragments matched against `Charge Type` and `Charge Description` **combined into one string**, not tried separately field-by-field. (An earlier version tried `Charge Type` to completion first and fell back to `Charge Description` only if nothing matched — that badly undercounted subcategories like "Ground", because a generic `Charge Type` value like "Freight" would match the catch-all "Transportation / Base Charge" pattern before ever looking at `Charge Description`, where the real detail — e.g. "Ground Commercial" — actually lives.) Order still matters within the combined text: more specific subcategories/categories are listed before generic ones, so e.g. "Chargeback Fuel Surcharge" is caught by Fuel Surcharge before the generic "chargeback" rule in Discounts & Credits gets a chance, and "Ground" is caught before the generic "Transportation / Base Charge" fallback.

In [ ]:
TAXONOMY = {
    "Fuel Surcharge": {
        "Fuel Surcharge Correction": [r"correction.*fuel", r"fuel.*correction"],
        "Chargeback Fuel Surcharge": [r"chargeback.*fuel"],
        "International Fuel Surcharge": [r"worldease.*fuel", r"international.*fuel"],
        "Domestic Fuel Surcharge": [r"\bfuel surcharge\b", r"\bfuel\b", r"brandstof"],
    },
    "Accessorial / Delivery Surcharge": {
        "Delivery Area Surcharge (DAS)": [r"\bdas\b", r"delivery area surcharge"],
        "Residential Delivery/Surcharge": [r"residential"],
        "Signature Required": [r"signature"],
        "Additional Handling": [r"add'?l handling", r"additional handling", r"handling.*dimension"],
        "Saturday / After Hours": [r"saturday", r"after hours"],
        "Demand / Peak Surcharge": [r"demand surcharge", r"surge emergency", r"\bpeak\b"],
        "Wait Time / Mileage / Toll": [r"wait time", r"additional miles", r"\btoll\b"],
        "Address Correction": [r"address correction"],
        "Security Surcharge": [r"security surcharge"],
    },
    "Discounts & Credits": {
        "Earned Discount": [r"earned discount"],
        "Grace Discount": [r"grace discount"],
        "General Discount": [r"\bdiscount\b"],
        "Credit From Carrier": [r"credit from carrier"],
        "Chargeback / Reversal": [r"\bchargeback\b"],
        "Billing Adjustment / Correction": [r"billing adjustment", r"shipping charge correction", r"verzendcorrectiekosten"],
    },
    "Taxes & Customs": {
        "VAT": [r"\bvat\b", r"\bbtw\b"],
        "GST / HST": [r"\bgst\b", r"\bhst\b"],
        "Duty & Import Tax": [r"dut(y|ies)", r"import fee", r"import tax"],
        "Customs / Brokerage": [r"customs?", r"brokerage", r"export declaration", r"international processing", r"internationale verwerkingskosten", r"\beei\b"],
        "Sales / General Tax": [r"\btax(es)?\b"],
    },
    "Administrative & Service Fees": {
        "Disbursement Fee": [r"disbursement"],
        "Third Party Billing": [r"third party billing"],
        "Document Fee": [r"document fee", r"documents? preparation"],
        "Package Handling / Storage": [r"package handling", r"warehouse storage", r"\bstorage\b"],
        "Pickup Service": [r"pickup"],
    },
    "Base Freight / Transportation": {
        "Ground": [r"\bground\b", r"\bltl\b", r"truckload", r"road ?freight"],
        "Domestic Air (Next/2nd/3 Day)": [r"next day", r"2nd day", r"two day", r"3 day", r"third day", r"second day"],
        "International / Export / Import Freight": [r"worldwide express", r"ww express", r"\bexport\b", r"\bimport\b", r"world ?ease"],
        "Ocean Freight": [r"ocean"],
        "Line Haul": [r"line haul"],
        "Transportation / Base Charge": [r"transportation charge", r"\bbase\b", r"\bfreight\b", r"frt freight"],
    },
}

CATCH_ALL = ("Other / Uncategorized", "Unclassified")

# Checked BEFORE the main taxonomy loop. Without this, a combo like Charge Type="Freight" /
# Charge Description="Ground Residential" gets caught by Accessorial's generic `residential`
# pattern before Base Freight is ever reached -- but "Ground Residential" / "Next Day Air
# Residential" etc. are freight *service-tier* names (the residential-rate version of that
# service), not a standalone residential-delivery surcharge line.
SERVICE_TIER_OVERRIDES = [
    (r"\bground\b.*\bresidential\b", ("Base Freight / Transportation", "Ground")),
    (r"\b(next day|2nd day|second day|3 day|third day)\b.*\bresidential\b", ("Base Freight / Transportation", "Domestic Air (Next/2nd/3 Day)")),
]

n_major = len(TAXONOMY) + 1  # + catch-all
print(f"{n_major} major categories (incl. catch-all)")
for major, subcats in TAXONOMY.items():
    print(f"  {major}: {len(subcats)} subcategories")

## Rule-based classifier

In [ ]:
def normalize(text):
    if pd.isna(text):
        return ""
    return str(text).lower().strip()


def rule_classify_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None
    for pat, result in SERVICE_TIER_OVERRIDES:
        if re.search(pat, t):
            return result
    for major, subcats in TAXONOMY.items():
        for sub, patterns in subcats.items():
            for pat in patterns:
                if re.search(pat, t):
                    return (major, sub)
    return None

## Apply to unique (Charge Type, Charge Description) combinations

Classifying at the unique-combination level (not per row) is what keeps this fast and reusable at any data volume.

In [ ]:
unique_labels = (
    detail.groupby(["Charge Type", "Charge Description"], dropna=False)["Charge Value"]
    .agg(row_count="size", total_value="sum")
    .reset_index()
)
print(f"{len(unique_labels):,} unique (Charge Type, Charge Description) combinations covering {unique_labels['row_count'].sum():,} rows")

classified = unique_labels.apply(lambda r: rule_classify_row(r["Charge Type"], r["Charge Description"]), axis=1)
unique_labels["Major Category"] = [c[0] if c else None for c in classified]
unique_labels["Subcategory"] = [c[1] if c else None for c in classified]
unique_labels["Method"] = ["rule" if c else None for c in classified]

matched_rows = unique_labels.loc[unique_labels["Major Category"].notna(), "row_count"].sum()
print(f"Rule-based coverage: {matched_rows / unique_labels['row_count'].sum() * 100:.1f}% of rows")

## TF-IDF fallback for labels the rules missed

For any (Charge Type, Charge Description) combo still unclassified, compare it against a small reference "document" built from each subcategory's own keywords, using character n-gram TF-IDF (robust to typos, pluralization, and non-English spellings) + cosine similarity. Assign to the best match if similarity clears a threshold, otherwise leave it in the catch-all bucket.

**Caveat found during QA** (see `charge_categorization_check.ipynb`): this fallback can score a *wrong* match confidently (0.4–0.7 similarity, well above the 0.25 threshold) when a label shares one generic word with a reference doc — e.g. "International Processing Fee" matched "International Fuel Surcharge" at 0.66 similarity purely because both strings contain "international", even though the charge has nothing to do with fuel. A handful of these were caught and moved into explicit rules above (`international processing`, `custom(s)? clearance`, `documents? preparation`, `eei`), but any *new* fallback match should still be treated as a suggestion to review, not an auto-accepted answer — a high similarity score alone doesn't guarantee it's correct.

In [ ]:
ref_docs, ref_labels = [], []
for major, subcats in TAXONOMY.items():
    for sub, patterns in subcats.items():
        doc = " ".join(p.replace(r"\b", "").replace(".*", " ").replace("'?", "").replace("?", "") for p in patterns)
        ref_docs.append(doc)
        ref_labels.append((major, sub))

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
ref_vectors = vectorizer.fit_transform(ref_docs)

SIMILARITY_THRESHOLD = 0.25


def fallback_classify_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None
    vec = vectorizer.transform([t])
    sims = cosine_similarity(vec, ref_vectors)[0]
    best_idx = sims.argmax()
    if sims[best_idx] >= SIMILARITY_THRESHOLD:
        return ref_labels[best_idx]
    return None


unmatched = unique_labels["Major Category"].isna()
for idx in unique_labels.index[unmatched]:
    row = unique_labels.loc[idx]
    result = fallback_classify_row(row["Charge Type"], row["Charge Description"])
    if result:
        unique_labels.loc[idx, "Major Category"] = result[0]
        unique_labels.loc[idx, "Subcategory"] = result[1]
        unique_labels.loc[idx, "Method"] = "tfidf_fallback"

unique_labels["Major Category"] = unique_labels["Major Category"].fillna(CATCH_ALL[0])
unique_labels["Subcategory"] = unique_labels["Subcategory"].fillna(CATCH_ALL[1])
unique_labels["Method"] = unique_labels["Method"].fillna("none")

final_coverage = 1 - unique_labels.loc[unique_labels["Major Category"] == CATCH_ALL[0], "row_count"].sum() / unique_labels["row_count"].sum()
print(f"Final coverage after TF-IDF fallback: {final_coverage * 100:.1f}% of rows categorized")
print(unique_labels["Method"].value_counts())

## Summary: rows and $ value by category

In [ ]:
summary = (
    unique_labels.groupby(["Major Category", "Subcategory"])
    .agg(row_count=("row_count", "sum"), total_value=("total_value", "sum"))
    .sort_values("row_count", ascending=False)
)
summary

In [ ]:
major_summary = (
    unique_labels.groupby("Major Category")
    .agg(row_count=("row_count", "sum"), total_value=("total_value", "sum"))
    .sort_values("row_count", ascending=False)
)
major_summary["pct_of_rows"] = (major_summary["row_count"] / major_summary["row_count"].sum() * 100).round(1)
major_summary

## Review the catch-all bucket

Anything left in "Other / Uncategorized" is worth a manual look — either add a new keyword pattern to `TAXONOMY`, or it genuinely doesn't fit the taxonomy yet.

In [ ]:
uncategorized = unique_labels[unique_labels["Major Category"] == CATCH_ALL[0]].sort_values("row_count", ascending=False)
print(f"{len(uncategorized):,} uncategorized combinations, {uncategorized['row_count'].sum():,} rows")
uncategorized.head(30)

## Save the reusable mapping table

This table is the reusable artifact: to classify a new/larger dataset, get its distinct (Charge Type, Charge Description) pairs, look them up here (or re-run `rule_classify_row` / `fallback_classify_label` on whatever's new), and merge the result back onto the full table — the expensive part (classification) never has to run per-row.

In [ ]:
unique_labels.to_csv("outputs/charge_category_mapping.csv", index=False)
print("Saved outputs/charge_category_mapping.csv")